In [1]:
import faiss
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain.schema import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
# from sentence_transformers import SentenceTransformer

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import os
from time import time
from tqdm import tqdm
import datetime

# cuda 12.4 에 맞춘 torch 환경 구성 (poetry로 진행하는 방법)
# poetry source add --priority=explicit pytorch-gpu-src https://download.pytorch.org/whl/cu124
# poetry add --source pytorch-gpu-src torch torchvision torchaudio

In [2]:
# https://smith.langchain.com
from langchain_teddynote import logging

logging.langsmith("WebCrawl2FAISS_DB")

LangSmith 추적을 시작합니다.
[프로젝트명]
WebCrawl2FAISS_DB


In [3]:
law_name = ["조세특례제한법 시행규칙", "소득세법 시행령", "국세기본법 시행령", "부가가치세법 시행령", "조세특례제한법 시행령", "법인세법 시행령", "종합부동산세법 시행령", "법인세법 시행규칙", "교통ㆍ에너지ㆍ환경세법 시행령", "개별소비세법 시행령", "상속세 및 증여세법", "상속세 및 증여세법 시행령", "소득세법 시행규칙", "종합부동산세법 시행규칙", "국제조세조정에 관한 법률", "국세징수법", "조세특례제한법", "소득세법", "부가가치세법"]
law_efYd = ["20241203", "20241112", "20241112", "20241112", "20241112", "20241112", "20241112", "20241111", "20241101", "20241101", "20240915", "20240915", "20240910", "20240910", "20240719", "20240719", "20240710", "20240701", "20240701"]
law_lsiSeq = ["266927", "266275", "266353", "266375", "266277", "266373", "266279", "266387", "266185", "266195", "254809", "265217", "265265", "265267", "252779", "252777", "258533", "258015", "257973"]
# baseURL = f"https://www.law.go.kr/lsInfoP.do?lsiSeq={lsiSeq}&efYd={efYd}#0000"

In [ ]:
# 창 안 띄우는 옵션
options = webdriver.ChromeOptions()
options.add_argument('--headless')
driver = webdriver.Chrome(options=options)
# 텍스트 분할 정의
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
# 허깅페이스 임베딩 정의
model_name = "intfloat/multilingual-e5-large-instruct"
hf_embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True},
    multi_process=True
)
# FAISS Vector DB 선언
dimension_size = len(hf_embeddings.embed_query("hello world"))
print(dimension_size)
db = FAISS(
    embedding_function=hf_embeddings,
    index=faiss.IndexFlatL2(dimension_size),
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)
# 각종 법률 웹크롤링 및 DB에 저장 시작
start_time = time()
name = law_name[0]
efYd = law_efYd[0]
lsiSeq = law_lsiSeq[0]
baseURL = f"https://www.law.go.kr/lsInfoP.do?lsiSeq={lsiSeq}&efYd={efYd}#0000"
driver.get(baseURL)
# "#conScroll .pgroup" 요소 로딩을 기다린다.
WebDriverWait(driver, 30).until(
    EC.presence_of_element_located((By.CSS_SELECTOR, "#conScroll .pgroup"))
)
# 1. 메인이 되는 법률 내용들을 크롤링하여, document 객체들의 리스트로 만든다.
eachItem = driver.find_elements(By.CSS_SELECTOR, "#conScroll .pgroup")
chapter = ''
passage = ''
article = ''
idx = 0
for item in tqdm(eachItem):
    t = item.text
    if t.startswith('^제.장'):
        chapter = t
        passage = ''
        article = ''
    elif t.startswith('^제.절'):
        passage = t
        article = ''
    else:
        print("-"*50)
        article = t
        doc = [Document(page_content=article, metadata={"law":name,"date":datetime.date(int(efYd[:4]),int(efYd[4:6]),int(efYd[6:])),"chapter":chapter,"passage":passage})]
        docs = text_splitter.split_documents(doc)
        # 2. 텍스트 분할을 진행한다. 각 절에 따른 데이터가 많지 않을 것 같아 생략해도 될 수도 있다. (개인적 판단이므로 우선 진행)
        start = time()
        split_documents = text_splitter.split_documents(docs)
        print(f"Success:text_split - {time()-start:.3f}")
        n = len(split_documents)
        print(f"len(split_documents) = {n}")
        # 3. Vector DB에 저장
        start = time()
        db.add_documents(split_documents, ids=[f"{name}_{num}" for num in range(idx, idx+n)])
        print(f"Success:add_documents - {time()-start:.3f}")
        idx += n
        print("-"*50)
driver.quit()
# 로컬 Disk 에 Vector DB 저장
db.save_local(folder_path="faiss_db", index_name="only266927")
# 총 걸린 시간 출력
print(time()-start_time)

In [5]:
'''
# 창 안 띄우는 옵션
options = webdriver.ChromeOptions()
options.add_argument('--headless')
driver = webdriver.Chrome(options=options)
# 텍스트 분할 정의
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
# 허깅페이스 임베딩 정의
model_name = "intfloat/multilingual-e5-large-instruct"
hf_embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True},
    multi_process=True
)
# FAISS Vector DB 선언
dimension_size = len(hf_embeddings.embed_query("hello world"))
print("dim:",dimension_size)
db = FAISS(
    embedding_function=hf_embeddings,
    index=faiss.IndexFlatL2(dimension_size),
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)
# 각종 법률 웹크롤링 및 DB에 저장 시작
start_time = time()
for name, efYd, lsiSeq in zip(law_name, law_efYd, law_lsiSeq):
    print(name)
    baseURL = f"https://www.law.go.kr/lsInfoP.do?lsiSeq={lsiSeq}&efYd={efYd}#0000"
    try:
        driver.get(baseURL)
        # "#conScroll .pgroup" 요소 로딩을 기다린다.
        WebDriverWait(driver, 30).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "#conScroll .pgroup"))
        )

        # 1. 메인이 되는 법률 내용들을 크롤링하여, document 객체들의 리스트로 만든다.
        eachItem = driver.find_elements(By.CSS_SELECTOR, "#conScroll .pgroup")
        chapter = ''
        passage = ''
        article = ''
        idx = 0
        for item in tqdm(eachItem):
            t = item.text
            if t.startswith('^제.장'):
                chapter = t
                passage = ''
                article = ''
            elif t.startswith('^제.절'):
                passage = t
                article = ''
            else:
                print("-"*50)
                article = t
                doc = [Document(page_content=article, metadata={"law":name,"date":datetime.date(int(efYd[:4]),int(efYd[4:6]),int(efYd[6:])),"chapter":chapter,"passage":passage})]
                docs = text_splitter.split_documents(doc)
                # 2. 텍스트 분할을 진행한다. 각 절에 따른 데이터가 많지 않을 것 같아 생략해도 될 수도 있다. (개인적 판단이므로 우선 진행)
                start = time()
                split_documents = text_splitter.split_documents(docs)
                print(f"Success:text_split - {time()-start:.3f}")
                n = len(split_documents)
                print(f"len(split_documents) = {n}")
                # 3. Vector DB에 저장
                start = time()
                db.add_documents(split_documents, ids=[f"{name}_{num}" for num in range(idx, idx+n)])
                print(f"Success:add_documents - {time()-start:.3f}")
                idx += n
                print("-"*50)
    except Exception as e:
        print(f"Error occurred while processing {name}: {e}")
        continue
# 로컬 Disk 에 Vector DB 저장
db.save_local(folder_path="faiss_db", index_name="lawDB")
# 총 걸린 시간 출력
spend_time = time()-start_time
spend_hour = spend_time//3600
spend_time -= spend_hour*3600
spend_minute = spend_time//60
spend_second = spend_time-(spend_minute*60)
print(f"총 소요 시간 : {int(spend_hour)}h {int(spend_minute):02d}m {spend_second:2.3f}sec")
'''

'\n# 창 안 띄우는 옵션\noptions = webdriver.ChromeOptions()\noptions.add_argument(\'--headless\')\ndriver = webdriver.Chrome(options=options)\n# 텍스트 분할 정의\ntext_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)\n# 허깅페이스 임베딩 정의\nmodel_name = "intfloat/multilingual-e5-large-instruct"\nhf_embeddings = HuggingFaceEmbeddings(\n    model_name=model_name,\n    model_kwargs={"device": "cuda"},\n    encode_kwargs={"normalize_embeddings": True},\n)\n# FAISS Vector DB 선언\ndimension_size = len(hf_embeddings.embed_query("hello world"))\nprint(dimension_size)\ndb = FAISS(\n    embedding_function=hf_embeddings,\n    index=faiss.IndexFlatL2(dimension_size),\n    docstore=InMemoryDocstore(),\n    index_to_docstore_id={},\n)\n# 각종 법률 웹크롤링 및 DB에 저장 시작\nstart_time = time()\nfor name, efYd, lsiSeq in zip(law_name, law_efYd, law_lsiSeq):\n    print(name)\n    baseURL = f"https://www.law.go.kr/lsInfoP.do?lsiSeq={lsiSeq}&efYd={efYd}#0000"\n    try:\n        driver.get(baseURL)\n        # "